<a href="https://colab.research.google.com/github/kyungeunvoyage/NailFoldExp/blob/main/(Analysis)AbsoluteThresholdDetction_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Basic Start

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# 1. 모든 AbsoluteThresholdDetection.csv 파일 자동 로드 및 통합
file_pattern = 'P*_AbsoluteThresholdDetection.csv'
all_files = glob.glob(file_pattern)
print(f"로드된 파일 개수: {len(all_files)}")

if not all_files:
    print("분석할 CSV 파일을 찾을 수 없습니다.")
else:
    print(f"발견된 파일: {all_files}")
    df_list = [pd.read_csv(f) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)

    # 2. 데이터 전처리
    # 공백 제거
    df['Condition'] = df['Condition'].str.strip()

    # Condition 명칭 통일 및 불필요한 조건 제거
    df['Condition'] = df['Condition'].replace('Active', 'On-touch (Mid)')
    df['Condition'] = df['Condition'].replace('On-touch (Hard)', 'On-touch (Mid)')
    df = df[df['Condition'] != 'On-touch (Soft)']

    # 분석 대상 구역 필터링 (A-F)
    df = df[df['Area'].isin(['A', 'B', 'C', 'D', 'E', 'F'])]

    # Force 문자열에서 숫자 추출 (예: '0.07g' -> 0.07)
    df['Force_Val'] = df['Force'].str.extract(r'(\d+\.?\d*)').astype(float)
    force_order = sorted(df['Force_Val'].unique())

    # 3. 시각화 (1x2 Plot)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.set_theme(style="white")

    # --- (Left) Psychometric Curve (전체 평균) ---
    # 조건/강도별 전체 평균 정답률 계산
    overall_acc = df.groupby(['Condition', 'Force_Val'])['IsCorrect'].mean().reset_index()

    sns.lineplot(ax=axes[0], data=overall_acc, x='Force_Val', y='IsCorrect',
                 hue='Condition', marker='o', linewidth=3, markersize=8)
    axes[0].axhline(0.8, color='red', linestyle='--', label='80% Threshold')
    axes[0].set_title('Detection Accuracy Curve', fontsize=15, fontweight='bold')
    axes[0].set_ylim(-0.05, 1.05)
    axes[0].legend()

    # --- (Right) Binary Accuracy Box Plot by Force ---
    sns.boxplot(
        ax=axes[1], data=df, x='Force_Val', y='IsCorrect', hue='Condition',
        order=force_order, palette=['#FFFFFF', '#D3D3D3'],
        linewidth=1.5, fliersize=0, width=0.6,
        medianprops={'color': 'red', 'linewidth': 2}
    )

    # 개별 응답 점 표시
    sns.stripplot(
        ax=axes[1], data=df, x='Force_Val', y='IsCorrect', hue='Condition',
        order=force_order, dodge=True, palette=['#000000', '#000000'],
        alpha=0.3, size=4, jitter=0.2
    )

    axes[1].set_title('Binary Accuracy Distribution by Force', fontsize=15, fontweight='bold')
    axes[1].set_xlabel('Stimulus Force (g)', fontsize=12)
    axes[1].set_ylabel('Accuracy (0=Wrong, 1=Correct)', fontsize=12)
    axes[1].set_ylim(-0.1, 1.1)

    # 레전드 정리
    handles, labels = axes[1].get_legend_handles_labels()
    axes[1].legend(handles[0:2], labels[0:2], title='Condition', frameon=False, loc='lower right')

    plt.tight_layout()
    plt.show()

    # 4. 통계 요약 출력
    print("\n" + "="*40)
    print("       [SUMMARY STATISTICS]       ")
    print("="*40)

    # 역치 산출 (정답률 80% 기준 최소 강도)
    print("\n1. Estimated Absolute Thresholds (80% Accuracy):")
    for cond in overall_acc['Condition'].unique():
        cond_df = overall_acc[overall_acc['Condition'] == cond]
        threshold_df = cond_df[cond_df['IsCorrect'] >= 0.8]

        if not threshold_df.empty:
            threshold = threshold_df['Force_Val'].min()
            print(f" - {cond}: {threshold}g")
        else:
            print(f" - {cond}: 80% 정답률에 도달하지 못함")

    # 구역별/강도별 상세 정답률 테이블
    area_acc = df.groupby(['Condition', 'Area'])['IsCorrect'].mean().reset_index()

    print("\n2. Regional Accuracy Table:")
    pivot_area = area_acc.pivot(index='Area', columns='Condition', values='IsCorrect')
    print(pivot_area)

    print("\n3. Force-wise Accuracy Table:")
    pivot_force = overall_acc.pivot(index='Force_Val', columns='Condition', values='IsCorrect')
    print(pivot_force)

#Accuracy Box Plot (Relative Accuracy (1 - |Error|/Target)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# 1. 모든 AbsoluteThresholdDetection.csv 파일 자동 로드 및 통합
file_pattern = 'P*_AbsoluteThresholdDetection.csv'
all_files = glob.glob(file_pattern)
print(f"로드된 파일 개수: {len(all_files)}")

if not all_files:
    print("분석할 CSV 파일을 찾을 수 없습니다.")
else:
    print(f"발견된 파일: {all_files}")
    df_list = [pd.read_csv(f) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)

    # 2. 데이터 전처리
    # 공백 제거 및 조건 명칭 통일
    df['Condition'] = df['Condition'].str.strip()
    df['Condition'] = df['Condition'].replace('Active', 'On-touch (Mid)')
    df['Condition'] = df['Condition'].replace('On-touch (Hard)', 'On-touch (Mid)')

    # 분석 제외 조건 필터링
    df = df[df['Condition'] != 'On-touch (Soft)']

    # 분석 대상 구역 필터링 (A~F)
    df = df[df['Area'].isin(['A', 'B', 'C', 'D', 'E', 'F'])]

    # Force 문자열에서 숫자만 추출 (예: '0.07g' -> 0.07)
    df['Force_Val'] = df['Force'].str.extract(r'(\d+\.?\d*)').astype(float)
    force_order = sorted(df['Force_Val'].unique())

    # 3. 새로운 Relative Accuracy 계산 (Target 대비 오차 비율 반영)
    def calc_relative_accuracy(row):
        # Target이 0인 경우(분모가 0이 되는 경우) 방지
        if row['Target'] == 0:
            return 100 if row['Response'] == 0 else 0

        # Target 대비 절대 오차 비율 계산: |Target - Response| / Target
        error_ratio = abs(row['Target'] - row['Response']) / row['Target']

        # 정확도 계산: (1 - 오차비율) * 100
        score = (1 - error_ratio) * 100

        # 점수가 음수가 되지 않도록 하한선(0) 적용
        return max(0, score)

    df['Relative_Score'] = df.apply(calc_relative_accuracy, axis=1)

    # 4. 시각화 (동일한 스타일 유지)
    plt.figure(figsize=(12, 7))
    sns.set_theme(style="white")

    # 박스플롯: 조건별(In-air vs On-touch) 비교
    ax = sns.boxplot(
        data=df, x='Force_Val', y='Relative_Score', hue='Condition',
        palette=['#FFFFFF', '#D3D3D3'],
        linewidth=1.5, fliersize=0, width=0.6, order=force_order,
        medianprops={'color': 'red', 'linewidth': 2}
    )

    # 개별 데이터 점(stripplot) 표시
    sns.stripplot(
        data=df, x='Force_Val', y='Relative_Score', hue='Condition',
        dodge=True, palette=['#000000', '#000000'],
        alpha=0.4, size=5, jitter=0.2, ax=ax, order=force_order
    )

    # 80% 성능 가이드라인
    plt.axhline(80, color='red', linestyle='--', linewidth=1.2, alpha=0.7)

    # 타이틀 및 라벨 설정
    num_subjects = df['SubjectID'].nunique() if 'SubjectID' in df.columns else len(all_files)
    plt.title(f'Absolute Threshold Detection: Relative Accuracy (N={num_subjects})\n(1 - |Error|/Target)', fontsize=16, pad=20)
    plt.xlabel('Stimulus Force (g)', fontsize=13)
    plt.ylabel('Relative Accuracy (%)', fontsize=13)
    plt.ylim(-5, 105)

    # 범례 설정 (중복 제거)
    handles, labels = ax.get_legend_handles_labels()
    plt.legend(handles[0:2], labels[0:2], title='Condition', frameon=False, loc='lower right')

    sns.despine()
    plt.tight_layout()
    plt.show()

    print("분석 및 시각화 완료.")

#Region 별 Accuracy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# 1. 모든 AbsoluteThresholdDetection.csv 파일 자동 로드 및 통합
file_pattern = 'P*_AbsoluteThresholdDetection.csv'
all_files = glob.glob(file_pattern)
print(f"로드된 파일 개수: {len(all_files)}")

if not all_files:
    print("분석할 CSV 파일을 찾을 수 없습니다. 파일 패턴을 확인해주세요.")
else:
    print(f"발견된 파일: {all_files}")
    df_list = [pd.read_csv(f) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)

    # 2. 데이터 전처리 및 클리닝
    # 공백 제거
    df['Condition'] = df['Condition'].str.strip()

    # Condition 명칭 통일 및 불필요한 조건 제거
    df['Condition'] = df['Condition'].replace('Active', 'On-touch (Mid)')
    df['Condition'] = df['Condition'].replace('On-touch (Hard)', 'On-touch (Mid)')
    df = df[df['Condition'] != 'On-touch (Soft)']

    # 분석 대상 구역 필터링 (A-F)
    df = df[df['Area'].isin(['A', 'B', 'C', 'D', 'E', 'F'])]

    # Force 문자열에서 숫자만 추출 (예: '0.07g' -> 0.07)
    df['Force_Val'] = df['Force'].str.extract(r'(\d+\.?\d*)').astype(float)

    # 3. 구역별/강도별 정답률 계산 (평균 곡선을 그리기 위함)
    # IsCorrect 컬럼을 기준으로 평균을 계산합니다.
    region_acc = df.groupby(['Condition', 'Area', 'Force_Val'])['IsCorrect'].mean().reset_index()

    print("✅ 데이터 전처리 및 Region별 그룹화 완료")

    # 4. 시각화 (1x2 Subplots: In-air vs On-touch)
    fig, axes = plt.subplots(1, 2, figsize=(20, 9), sharey=True)
    sns.set_theme(style="whitegrid")

    conditions = ['In-air', 'On-touch (Mid)']
    # 구역별 색상 고정 (A-F)
    colors = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#2ca02c", "D": "#FF69B4", "E": "#9467bd", "F": "#d62728"}
    area_order = ['A', 'B', 'C', 'D', 'E', 'F']

    for i, cond in enumerate(conditions):
        cond_data = region_acc[region_acc['Condition'] == cond]

        # Area별로 선 그래프(Lineplot) 생성
        sns.lineplot(
            ax=axes[i],
            data=cond_data,
            x='Force_Val',
            y='IsCorrect',
            hue='Area',
            hue_order=area_order,
            palette=colors,
            marker='o',
            linewidth=2.5,
            markersize=9
        )

        # 기준선 추가 (80% 역치 및 50% 기회 수준)
        axes[i].axhline(0.8, color='red', linestyle='--', alpha=0.6, label='80% Threshold')
        axes[i].axhline(0.5, color='black', linestyle=':', alpha=0.8, label='Chance Level (0.5)')

        # 타이틀 및 라벨 설정
        num_subjects = len(all_files)
        axes[i].set_title(f'Detection Sensitivity: {cond} (N={num_subjects})', fontsize=18, fontweight='bold')
        axes[i].set_xlabel('Stimulus Force (g)', fontsize=14)
        axes[i].set_ylabel('Accuracy (Mean)' if i == 0 else "", fontsize=14)
        axes[i].set_ylim(-0.05, 1.05)

        # 범례 설정
        axes[i].legend(title="Finger Region", fontsize=11, frameon=False, loc='lower right')

    sns.despine()
    plt.tight_layout()
    plt.show()

    # 간단한 요약 출력
    print("\n[80% 정확도 도달 최소 강도 요약]")
    for cond in conditions:
        for area in area_order:
            sub = region_acc[(region_acc['Condition'] == cond) & (region_acc['Area'] == area)]
            threshold_met = sub[sub['IsCorrect'] >= 0.8]
            if not threshold_met.empty:
                min_force = threshold_met['Force_Val'].min()
                print(f" - {cond} | Region {area}: {min_force}g")
            else:
                print(f" - {cond} | Region {area}: 도달 못함")

#Box Plot for Region (x-axis is Force)  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# 1. 모든 AbsoluteThresholdDetection.csv 파일 자동 로드 및 통합
file_pattern = 'P*_AbsoluteThresholdDetection.csv'
all_files = glob.glob(file_pattern)

if not all_files:
    print("분석할 CSV 파일을 찾을 수 없습니다. 파일명을 확인해주세요.")
else:
    print(f"발견된 파일: {all_files}")
    df_list = [pd.read_csv(f) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)

    # 2. 데이터 전처리
    # Force 값 추출 (예: '0.6g' -> 0.6)
    df['Force_Val'] = df['Force'].str.extract(r'(\d+\.?\d*)').astype(float)

    # [수정된 부분] 3. 새로운 Relative Accuracy 계산 (Target 대비 오차 비율 반영)
    def calc_relative_accuracy(row):
        # Target이 0인 경우(분모가 0) 방지
        if row['Target'] == 0:
            return 100 if row['Response'] == 0 else 0

        # Target 대비 절대 오차 비율 계산: |Target - Response| / Target
        error_ratio = abs(row['Target'] - row['Response']) / row['Target']

        # 정확도 계산: (1 - 오차비율) * 100
        score = (1 - error_ratio) * 100

        # 점수가 음수가 되지 않도록 하한선(0) 적용
        return max(0, score)

    df['Relative_Score'] = df.apply(calc_relative_accuracy, axis=1)

    # 4. 시각화 (1x2 Subplots: In-air vs On-touch)
    fig, axes = plt.subplots(1, 2, figsize=(22, 10), sharey=True)
    sns.set_theme(style="white")

    conditions = ['In-air', 'On-touch (Mid)']
    # 구역별 색상 설정 (D 구역 핑크 유지)
    colors = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#2ca02c", "D": "#FF69B4","E" : "#9467bd", "F": "#d62728"}
    force_order = sorted(df['Force_Val'].unique())

    for i, cond in enumerate(conditions):
        cond_data = df[df['Condition'] == cond]

        # 만약 데이터가 비어있다면 skip
        if cond_data.empty:
            axes[i].set_title(f'No data for {cond}')
            continue

        # Box Plot: x축은 Force, hue는 Area
        sns.boxplot(
            ax=axes[i],
            data=cond_data,
            x='Force_Val',
            y='Relative_Score',
            hue='Area',
            hue_order=['A', 'B', 'C', 'D','E','F'],
            palette=colors,
            width=0.7,
            linewidth=1.5,
            fliersize=0,
            order=force_order
        )

        # Scatter Point (Strip plot) 겹치기
        sns.stripplot(
            ax=axes[i],
            data=cond_data,
            x='Force_Val',
            y='Relative_Score',
            hue='Area',
            hue_order=['A', 'B', 'C', 'D','E','F'],
            dodge=True,
            palette=['#000000']*4, # 시인성을 위해 검은색 점 사용
            alpha=0.3,
            size=4,
            jitter=0.15,
            order=force_order,
            legend=False
        )

        # 80% 역치 기준선 (정확도 가이드라인)
        axes[i].axhline(80, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='80% Threshold')

        # 타이틀 및 라벨 설정
        num_subjects = len(all_files)
        axes[i].set_title(f'{cond}: Relative Accuracy (N={num_subjects})', fontsize=18, fontweight='bold')
        axes[i].set_xlabel('Stimulus Force (g)', fontsize=14)
        axes[i].set_ylabel('Accuracy (%) - (1 - |Error|/Target)' if i==0 else "", fontsize=14)
        axes[i].set_ylim(-5, 105)

        # 레전드 설정 (두 번째 플롯에만 표시)
        if i == 1:
            handles, labels = axes[i].get_legend_handles_labels()
            # Boxplot 핸들만 추출 (첫 4개 Area + 1개 Threshold 선)
            axes[i].legend(handles[0:5], labels[0:5], title="Finger Region", loc='lower right', frameon=False, fontsize=12)
        else:
            if axes[i].get_legend():
                axes[i].get_legend().remove()

    sns.despine()
    plt.tight_layout()
    plt.show()

    # 간단한 요약 통계 출력
    print("\n" + "="*50)
    print(" [Condition/Area별 평균 정확도(Relative Accuracy %)] ")
    print("="*50)
    summary = df.groupby(['Condition', 'Area'])['Relative_Score'].mean().unstack()
    print(summary)

#Box Plot Region (x-axis is Regions)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# 1. 모든 AbsoluteThresholdDetection.csv 파일 자동 로드 및 통합
file_pattern = 'P*_AbsoluteThresholdDetection.csv'
all_files = glob.glob(file_pattern)

if not all_files:
    print("분석할 CSV 파일을 찾을 수 없습니다.")
else:
    print(f"발견된 파일: {all_files}")
    df_list = [pd.read_csv(f) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)

    # 2. 데이터 전처리
    # Force 값 추출
    df['Force_Val'] = df['Force'].str.extract('(\d+\.?\d*)').astype(float)
    # Condition 명칭 정리
    df['Condition'] = df['Condition'].str.strip()
    df['Condition'] = df['Condition'].replace({'Active': 'On-touch (Mid)', 'On-touch (Hard)': 'On-touch (Mid)'})
    df = df[df['Condition'] != 'On-touch (Soft)']

    # 3. 페널티 로직 (기존 유지)
    def calc_relative_score(row):
        if row['Response'] == 0: return 0
        error = abs(row['Target'] - row['Response'])
        score = 100 - (error * 12.5)
        return max(0, score)

    df['Relative_Score'] = df.apply(calc_relative_score, axis=1)

    # 4. 시각화 (X축: Area, Hue: Force)
    fig, axes = plt.subplots(1, 2, figsize=(22, 10), sharey=True)
    sns.set_theme(style="white")

    conditions = ['In-air', 'On-touch (Mid)']
    # 강도별 색상을 시각적으로 구분하기 좋은 팔레트 설정 (강도가 높을수록 진해지게 설정 가능)
    force_order = sorted(df['Force_Val'].unique())
    area_order = ['A', 'B', 'C', 'D','E','F']

    # 강도 종류 수에 맞게 컬러맵 생성
    palette_force = sns.color_palette("viridis", len(force_order))

    for i, cond in enumerate(conditions):
        cond_data = df[df['Condition'] == cond]

        # Box Plot: x축은 Area, hue는 Force_Val
        sns.boxplot(
            ax=axes[i],
            data=cond_data,
            x='Area',
            y='Relative_Score',
            hue='Force_Val',
            order=area_order,
            hue_order=force_order,
            palette=palette_force,
            width=0.7,
            linewidth=1.5,
            fliersize=0
        )

        # Scatter Point (Strip plot) 겹치기
        sns.stripplot(
            ax=axes[i],
            data=cond_data,
            x='Area',
            y='Relative_Score',
            hue='Force_Val',
            order=area_order,
            hue_order=force_order,
            dodge=True,           # Box 위치에 맞춰 점 분산
            palette=['#000000'] * len(force_order), # 검은색 점
            alpha=0.3,
            size=4,
            jitter=0.15,
            legend=False
        )

        # 80% 역치 기준선
        axes[i].axhline(80, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='80% Threshold')

        # 타이틀 및 라벨 설정
        num_subjects = len(all_files)
        axes[i].set_title(f'{cond} (N={num_subjects})', fontsize=20, fontweight='bold')
        axes[i].set_xlabel('Finger Region', fontsize=14)
        axes[i].set_ylabel('Relative Correctness (%)' if i==0 else "", fontsize=14)
        axes[i].set_ylim(-5, 105)

        # 레전드 설정
        if i == 1:
            axes[i].legend(title="Stimulus Force (g)", loc='lower right', frameon=False, fontsize=12)
        else:
            if axes[i].get_legend():
                axes[i].get_legend().remove()

    sns.despine()
    plt.tight_layout()
    plt.show()

#Box Blot plus Line

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# 1. 모든 AbsoluteThresholdDetection.csv 파일 자동 로드 및 통합
file_pattern = 'P*_AbsoluteThresholdDetection.csv'
all_files = glob.glob(file_pattern)
print(f"로드된 파일 개수: {len(all_files)}")

if not all_files:
    print("분석할 CSV 파일을 찾을 수 없습니다.")
else:
    print(f"발견된 파일: {all_files}")
    df_list = [pd.read_csv(f) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)

    # 2. 데이터 전처리
    df['Force_Val'] = df['Force'].str.extract('(\d+\.?\d*)').astype(float)
    df['Condition'] = df['Condition'].str.strip()
    df['Condition'] = df['Condition'].replace('Active', 'On-touch (Mid)')
    df['Condition'] = df['Condition'].replace('On-touch (Hard)', 'On-touch (Mid)')
    df = df[df['Condition'] != 'On-touch (Soft)']

    # [수정된 부분] 3. 1번 방식 로직 적용 (1 - |Error|/Target)
    def calc_relative_score(row):
        # Response가 0이면(아무 응답도 못함) 0점 처리
        if row['Response'] == 0:
            return 0

        # Target이 0인 경우(분모 0 방지)
        if row['Target'] == 0:
            return 100 if row['Response'] == 0 else 0

        # Target 대비 절대 오차 비율 계산
        error_ratio = abs(row['Target'] - row['Response']) / row['Target']

        # 정확도 계산: (1 - 오차비율) * 100
        score = (1 - error_ratio) * 100

        # 점수가 음수가 되지 않도록 하한선(0) 적용
        return max(0, score)

    df['Relative_Score'] = df.apply(calc_relative_score, axis=1)

    # 4. 시각화 (1x2 Subplots)
    fig, axes = plt.subplots(1, 2, figsize=(22, 10), sharey=True)
    sns.set_theme(style="white")

    conditions = ['In-air', 'On-touch (Mid)']
    colors = {"A": "#1f77b4", "B": "#ff7f0e", "C": "#2ca02c", "D": "#FF69B4", "E": "#9467bd", "F": "#d62728"}
    force_order = sorted(df['Force_Val'].unique())
    area_order = ['A', 'B', 'C', 'D', 'E', 'F']

    for i, cond in enumerate(conditions):
        cond_data = df[df['Condition'] == cond]

        # [A] Box Plot: 투명도 0.3
        sns.boxplot(
            ax=axes[i],
            data=cond_data,
            x='Force_Val',
            y='Relative_Score',
            hue='Area',
            hue_order=area_order,
            palette=colors,
            width=0.7,
            linewidth=1.2,
            fliersize=0,
            order=force_order,
            boxprops=dict(alpha=0.3)
        )

        # [B] Line Plot (Point plot) 겹치기
        sns.pointplot(
            ax=axes[i],
            data=cond_data,
            x='Force_Val',
            y='Relative_Score',
            hue='Area',
            hue_order=area_order,
            palette=colors,
            order=force_order,
            markers='d',
            scale=0.8,
            linestyles='-',
            errorbar=None,
            dodge=0.55
        )

        # [C] Scatter Point (Strip plot) 겹치기
        sns.stripplot(
            ax=axes[i],
            data=cond_data,
            x='Force_Val',
            y='Relative_Score',
            hue='Area',
            hue_order=area_order,
            dodge=True,
            palette=['#000000']*6,
            alpha=0.2,
            size=3,
            jitter=0.1,
            order=force_order,
            legend=False
        )

        # 80% 역치 기준선
        axes[i].axhline(80, color='red', linestyle='--', linewidth=1.5, alpha=0.7, label='80% Threshold')

        num_subjects = len(all_files)
        axes[i].set_title(f'{cond} (N={num_subjects})', fontsize=20, fontweight='bold')
        axes[i].set_xlabel('Stimulus Force (g)', fontsize=14)
        # Y축 라벨을 정확도 계산식 명칭으로 변경
        axes[i].set_ylabel('Relative Accuracy (%) - (1 - |Error|/Target)' if i==0 else "", fontsize=14)
        axes[i].set_ylim(-5, 105)

        if i == 1:
            handles, labels = axes[i].get_legend_handles_labels()
            axes[i].legend(handles[0:6], labels[0:6], title="Finger Region", loc='lower right', frameon=False, fontsize=12)
        else:
            if axes[i].get_legend():
                axes[i].get_legend().remove()

    sns.despine()
    plt.tight_layout()
    plt.show()

#각 region 별로 box plot 그리고 위에 line

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os

# 1. 모든 AbsoluteThresholdDetection.csv 파일 자동 로드 및 통합
file_pattern = 'P*_AbsoluteThresholdDetection.csv'
all_files = glob.glob(file_pattern)
print(f"로드된 파일 개수: {len(all_files)}")

if not all_files:
    print("분석할 CSV 파일을 찾을 수 없습니다.")
else:
    print(f"발견된 파일: {all_files}")
    df_list = [pd.read_csv(f) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)

    # 2. 데이터 전처리
    df['Force_Val'] = df['Force'].str.extract('(\d+\.?\d*)').astype(float)
    df['Condition'] = df['Condition'].str.strip()
    df['Condition'] = df['Condition'].replace({'Active': 'On-touch (Mid)', 'On-touch (Hard)': 'On-touch (Mid)'})
    df = df[df['Condition'].isin(['In-air', 'On-touch (Mid)'])]

    # [수정된 부분] 3. 1번 방식 로직 적용 (1 - |Error|/Target)
    def calc_relative_score(row):
        # 아무 응답이 없는 경우 0점 처리
        if row['Response'] == 0:
            return 0

        # Target이 0인 경우(분모 0 방지) 처리
        if row['Target'] == 0:
            return 100 if row['Response'] == 0 else 0

        # Target 대비 절대 오차 비율 계산
        error_ratio = abs(row['Target'] - row['Response']) / row['Target']

        # 정확도 계산: (1 - 오차비율) * 100
        score = (1 - error_ratio) * 100

        # 점수가 음수가 되지 않도록 하한선(0) 적용
        return max(0, score)

    df['Relative_Score'] = df.apply(calc_relative_score, axis=1)

    # 4. 시각화 (3x2 Subplots: 각 Region별로 In-air vs On-touch 비교)
    fig, axes = plt.subplots(3, 2, figsize=(20, 24), sharey=True)
    axes = axes.flatten()
    sns.set_theme(style="whitegrid")

    cond_colors = {"In-air": "#1f77b4", "On-touch (Mid)": "#ff7f0e"}
    force_order = sorted(df['Force_Val'].unique())
    area_order = ['A', 'B', 'C', 'D', 'E', 'F']

    for i, reg in enumerate(area_order):
        reg_data = df[df['Area'] == reg]
        ax = axes[i]

        # [A] Box Plot: 투명도 0.3
        sns.boxplot(
            ax=ax,
            data=reg_data,
            x='Force_Val',
            y='Relative_Score',
            hue='Condition',
            palette=cond_colors,
            width=0.7,
            linewidth=1.2,
            fliersize=0,
            order=force_order,
            boxprops=dict(alpha=0.3)
        )

        # [B] Line Plot (Point plot): 평균 추세선
        sns.pointplot(
            ax=ax,
            data=reg_data,
            x='Force_Val',
            y='Relative_Score',
            hue='Condition',
            palette=cond_colors,
            order=force_order,
            markers='d',
            scale=0.8,
            linestyles='-',
            errorbar=None,
            dodge=0.45
        )

        # [C] Scatter Point (Strip plot)
        sns.stripplot(
            ax=ax,
            data=reg_data,
            x='Force_Val',
            y='Relative_Score',
            hue='Condition',
            dodge=True,
            palette=['#000000']*2,
            alpha=0.2,
            size=4,
            jitter=0.1,
            order=force_order,
            legend=False
        )

        # 80% 역치 기준선
        ax.axhline(80, color='red', linestyle='--', linewidth=1.5, alpha=0.7)

        # 타이틀 및 라벨 설정 (수식 표기 업데이트)
        num_subjects = len(all_files)
        ax.set_title(f'Region {reg}: In-air vs On-touch (N={num_subjects})', fontsize=18, fontweight='bold')
        ax.set_xlabel('Stimulus Force (g)', fontsize=14)
        ax.set_ylabel('Accuracy (%) - (1 - |Error|/Target)' if i % 2 == 0 else "", fontsize=14)
        ax.set_ylim(-5, 105)

        if i == 0:
            handles, labels = ax.get_legend_handles_labels()
            ax.legend(handles[0:2], labels[0:2], title="Condition", loc='lower right', frameon=False, fontsize=12)
        else:
            if ax.get_legend():
                ax.get_legend().remove()

    sns.despine()
    plt.tight_layout()
    plt.show()

#Psychometric Curve

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import os
from scipy.optimize import curve_fit

# 1. 파일 로드 및 통합
file_pattern = 'P*_AbsoluteThresholdDetection.csv'
all_files = glob.glob(file_pattern)

if not all_files:
    print("CSV 파일을 찾을 수 없습니다.")
else:
    df_list = [pd.read_csv(f) for f in all_files]
    df = pd.concat(df_list, ignore_index=True)
    df['Force_Val'] = df['Force'].str.replace('g', '').astype(float)

    # 2. 새로운 Relative Accuracy 계산 로직 반영
    def calc_relative_accuracy(row):
        if row['Target'] == 0:
            return 100 if row['Response'] == 0 else 0
        error_ratio = abs(row['Target'] - row['Response']) / row['Target']
        score = (1 - error_ratio) * 100
        return max(0, score)

    df['Relative_Score'] = df.apply(calc_relative_accuracy, axis=1)

    # 3. S-curve 피팅을 위한 집계 (Condition & Force별 평균 점수)
    # 0~100점을 0~1 범위로 정규화하여 피팅에 사용
    agg_df = df.groupby(['Condition', 'Force_Val'])['Relative_Score'].mean().reset_index()
    agg_df['Relative_Score_Norm'] = agg_df['Relative_Score'] / 100.0

    # 4. S-curve(Sigmoid) 함수 정의
    def sigmoid(x, L, x0, k, b):
        return b + (L - b) / (1 + np.exp(-k * (x - x0)))

    # --- 시각화 A: S-curve (Psychometric Function) ---
    plt.figure(figsize=(10, 6))
    conditions = agg_df['Condition'].unique()
    colors = ['#555555', '#AAAAAA'] # 조건 구분을 위한 무채색 톤

    for i, cond in enumerate(conditions):
        subset = agg_df[agg_df['Condition'] == cond]
        x_data = subset['Force_Val'].values
        y_data = subset['Relative_Score_Norm'].values

        # 초기값 설정 및 피팅
        p0 = [1.0, np.median(x_data), 5.0, 0.0]
        try:
            popt, _ = curve_fit(sigmoid, x_data, y_data, p0,
                                bounds=([0.7, 0, 0, 0], [1.0, 2.0, 30.0, 0.5]), maxfev=5000)
            x_fit = np.linspace(x_data.min(), x_data.max(), 100)
            y_fit = sigmoid(x_fit, *popt)

            plt.plot(x_fit, y_fit * 100, label=f'{cond} Fit', color=colors[i % 2], lw=2.5)
            plt.scatter(x_data, y_data * 100, color=colors[i % 2], edgecolors='black', s=60, zorder=3)
            print(f"[{cond}] Threshold (80% Accuracy): {popt[1]:.3f} g")
        except:
            plt.scatter(x_data, y_data * 100, label=f'{cond} (Raw Data)', alpha=0.5)

    plt.axhline(80, color='red', linestyle='--', linewidth=1.2, alpha=0.7, label='80% Performance')
    plt.title('Psychometric S-Curve: Relative Accuracy', fontsize=15)
    plt.xlabel('Stimulus Force (g)', fontsize=12)
    plt.ylabel('Relative Accuracy (%)', fontsize=12)
    plt.ylim(-5, 105)
    plt.legend()
    plt.grid(True, axis='y', linestyle=':', alpha=0.7)
    sns.despine()
    plt.show()

    # --- 시각화 B: 요청하신 Boxplot 스타일 ---
    plt.figure(figsize=(12, 7))
    sns.set_theme(style="white")
    force_order = sorted(df['Force_Val'].unique())

    ax = sns.boxplot(
        data=df, x='Force_Val', y='Relative_Score', hue='Condition',
        palette=['#FFFFFF', '#D3D3D3'],
        linewidth=1.5, fliersize=0, width=0.6, order=force_order,
        medianprops={'color': 'red', 'linewidth': 2}
    )

    sns.stripplot(
        data=df, x='Force_Val', y='Relative_Score', hue='Condition',
        dodge=True, palette=['#000000', '#000000'],
        alpha=0.4, size=5, jitter=0.2, ax=ax, order=force_order
    )

    plt.axhline(80, color='red', linestyle='--', linewidth=1.2, alpha=0.7)
    plt.title('Absolute Threshold Detection: Relative Accuracy (1 - |Error|/Target)', fontsize=16, pad=20)
    plt.xlabel('Stimulus Force (g)', fontsize=13)
    plt.ylabel('Relative Accuracy (%)', fontsize=13)
    plt.ylim(-5, 105)

    handles, labels = ax.get_legend_handles_labels()
    plt.legend(handles[0:2], labels[0:2], title='Condition', frameon=False, loc='lower right')
    sns.despine()
    plt.tight_layout()
    plt.show()